# Final Analysis — Drop Columns + Feature Engineering

Takes the median_version dataset, drops 11 redundant columns, creates 15 new features, and saves as H&M ready dataset (parquet).

In [1]:
import pandas as pd
import numpy as np
import os

INPUT_DIR = r'A:\IDEs\PyCharm\hAnd-m-PersonalizedFashionRecommendations\clean dataset\median_version'
OUTPUT_DIR = r'A:\IDEs\PyCharm\hAnd-m-PersonalizedFashionRecommendations\H&M ready dataset'
os.makedirs(OUTPUT_DIR, exist_ok=True)

articles = pd.read_parquet(os.path.join(INPUT_DIR, 'articles.parquet'))
customers = pd.read_parquet(os.path.join(INPUT_DIR, 'customers.parquet'))
transactions = pd.read_parquet(os.path.join(INPUT_DIR, 'transactions.parquet'))

print(f'Loaded: articles {articles.shape}, customers {customers.shape}, transactions {transactions.shape}')

Loaded: articles (105542, 25), customers (1371980, 7), transactions (31788324, 5)


## 1. Drop redundant columns from articles

11 columns removed — each has a name version that says the same thing.

In [2]:
drop_cols = [
    'product_code',           # redundant with article_id
    'prod_name',              # text version of product_type_name
    'graphical_appearance_no',# redundant with graphical_appearance_name
    'colour_group_code',      # redundant with colour_group_name
    'perceived_colour_value_id',     # redundant with perceived_colour_value_name
    'perceived_colour_master_id',    # redundant with perceived_colour_master_name
    'department_no',          # redundant with department_name
    'section_no',             # redundant with section_name
    'garment_group_no',       # redundant with garment_group_name
    'detail_desc',            # free text, needs NLP, low value for simple models
]

print(f'articles before: {articles.shape}')
articles = articles.drop(columns=drop_cols)
print(f'articles after:  {articles.shape}')
print(f'\nRemaining columns:')
for col in articles.columns:
    print(f'  {col}')

# Drop postal_code from customers
print(f'\ncustomers before: {customers.shape}')
customers = customers.drop(columns=['postal_code'])
print(f'customers after:  {customers.shape}')

articles before: (105542, 25)
articles after:  (105542, 15)

Remaining columns:
  article_id
  product_type_no
  product_type_name
  product_group_name
  graphical_appearance_name
  colour_group_name
  perceived_colour_value_name
  perceived_colour_master_name
  department_name
  index_code
  index_name
  index_group_no
  index_group_name
  section_name
  garment_group_name

customers before: (1371980, 7)
customers after:  (1371980, 6)


## 2. Feature engineering — transactions

In [3]:
# day_of_week — extract from t_dat
transactions['day_of_week'] = transactions['t_dat'].dt.day_name()
print('Created: day_of_week')

# month — extract from t_dat
transactions['month'] = transactions['t_dat'].dt.month
print('Created: month')

# is_weekend — 1 if Saturday/Sunday, 0 otherwise
transactions['is_weekend'] = (transactions['t_dat'].dt.dayofweek >= 5).astype(int)
print('Created: is_weekend')

# price_bracket — bin price into cheap/mid/expensive
transactions['price_bracket'] = pd.cut(transactions['price'], 
    bins=[-1, transactions['price'].quantile(0.33), transactions['price'].quantile(0.66), 999],
    labels=['cheap', 'mid', 'expensive'])
print('Created: price_bracket')

print(f'\ntransactions shape: {transactions.shape}')
print(transactions[['t_dat', 'price', 'day_of_week', 'month', 'is_weekend', 'price_bracket']].head())

Created: day_of_week
Created: month
Created: is_weekend
Created: price_bracket

transactions shape: (31788324, 9)
       t_dat     price day_of_week  month  is_weekend price_bracket
0 2018-09-20  0.050831    Thursday      9           0     expensive
1 2018-09-20  0.030492    Thursday      9           0           mid
2 2018-09-20  0.015237    Thursday      9           0         cheap
3 2018-09-20  0.016932    Thursday      9           0         cheap
4 2018-09-20  0.016932    Thursday      9           0         cheap


## 3. Feature engineering — customer-level features

In [4]:
# Group transactions per customer
cust_stats = transactions.groupby('customer_id').agg(
    customer_total_purchases=('article_id', 'count'),
    customer_unique_articles=('article_id', 'nunique'),
    price_avg_per_customer=('price', 'mean'),
).reset_index()

# days_since_last_purchase — days from max date to each customer's last purchase
max_date = transactions['t_dat'].max()
last_purchase = transactions.groupby('customer_id')['t_dat'].max().reset_index()
last_purchase['days_since_last_purchase'] = (max_date - last_purchase['t_dat']).dt.days
last_purchase = last_purchase[['customer_id', 'days_since_last_purchase']]

# repeat_purchase_rate — how often customer buys same article again
repeat = transactions.groupby(['customer_id', 'article_id']).size().reset_index(name='count')
repeat_rate = repeat.groupby('customer_id').agg(
    repeat_purchase_rate=('count', lambda x: (x > 1).sum() / len(x))
).reset_index()

# channel_preference — 1=online-only, 2=store-only, 3=both
channel = transactions.groupby('customer_id')['sales_channel_id'].nunique().reset_index()
channel['channel_preference'] = channel['sales_channel_id'].map({1: 'single', 2: 'both'})
channel = channel[['customer_id', 'channel_preference']]

# customer_top_category — most bought product_group_name per customer
tx_art = transactions.merge(articles[['article_id', 'product_group_name']], on='article_id', how='left')
top_cat = tx_art.groupby(['customer_id', 'product_group_name']).size().reset_index(name='cnt')
top_cat = top_cat.sort_values('cnt', ascending=False).drop_duplicates('customer_id')
top_cat = top_cat[['customer_id', 'product_group_name']].rename(columns={'product_group_name': 'customer_top_category'})

# Merge all customer features
customers = customers.merge(cust_stats, on='customer_id', how='left')
customers = customers.merge(last_purchase, on='customer_id', how='left')
customers = customers.merge(repeat_rate, on='customer_id', how='left')
customers = customers.merge(channel, on='customer_id', how='left')
customers = customers.merge(top_cat, on='customer_id', how='left')

# Fill NaN for customers with no purchases
customers['customer_total_purchases'] = customers['customer_total_purchases'].fillna(0)
customers['customer_unique_articles'] = customers['customer_unique_articles'].fillna(0)
customers['price_avg_per_customer'] = customers['price_avg_per_customer'].fillna(0)
customers['days_since_last_purchase'] = customers['days_since_last_purchase'].fillna(9999)
customers['repeat_purchase_rate'] = customers['repeat_purchase_rate'].fillna(0)
customers['channel_preference'] = customers['channel_preference'].fillna('none')
customers['customer_top_category'] = customers['customer_top_category'].fillna('none')

# age_group — bin age into ranges
customers['age_group'] = pd.cut(customers['age'], bins=[0, 20, 30, 40, 50, 60, 100],
    labels=['<20', '20-30', '30-40', '40-50', '50-60', '60+'])

print(f'customers shape: {customers.shape}')
print(f'\nNew columns:')
for col in ['age_group', 'customer_total_purchases', 'customer_unique_articles',
            'price_avg_per_customer', 'days_since_last_purchase', 'repeat_purchase_rate',
            'channel_preference', 'customer_top_category']:
    print(f'  {col}')
print(f'\nSample:')
print(customers[['customer_id', 'age', 'age_group', 'customer_total_purchases',
                  'days_since_last_purchase', 'channel_preference', 'customer_top_category']].head())

customers shape: (1371980, 14)

New columns:
  age_group
  customer_total_purchases
  customer_unique_articles
  price_avg_per_customer
  days_since_last_purchase
  repeat_purchase_rate
  channel_preference
  customer_top_category

Sample:
                                         customer_id   age age_group  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  49.0     40-50   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  25.0     20-30   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...  24.0     20-30   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...  54.0     50-60   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...  52.0     50-60   

   customer_total_purchases  days_since_last_purchase channel_preference  \
0                      21.0                      17.0               both   
1                      86.0                      76.0               both   
2                      18.0                       7.0             single   
3                      

## 4. Feature engineering — article-level features

In [5]:
# article_total_purchases — how many times this article was bought
# article_unique_customers — how many distinct customers bought it
art_stats = transactions.groupby('article_id').agg(
    article_total_purchases=('customer_id', 'count'),
    article_unique_customers=('customer_id', 'nunique'),
).reset_index()

articles = articles.merge(art_stats, on='article_id', how='left')

# Fill NaN for articles never bought
articles['article_total_purchases'] = articles['article_total_purchases'].fillna(0)
articles['article_unique_customers'] = articles['article_unique_customers'].fillna(0)

print(f'articles shape: {articles.shape}')
print(f'\nNew columns:')
for col in ['article_total_purchases', 'article_unique_customers']:
    print(f'  {col}')
print(f'\nSample:')
print(articles[['article_id', 'product_type_name', 'article_total_purchases', 'article_unique_customers']].head())

articles shape: (105542, 17)

New columns:
  article_total_purchases
  article_unique_customers

Sample:
  article_id product_type_name  article_total_purchases  \
0  108775015          Vest top                  10841.0   
1  108775044          Vest top                   7250.0   
2  108775051          Vest top                    215.0   
3  110065001               Bra                   1044.0   
4  110065002               Bra                    539.0   

   article_unique_customers  
0                    6885.0  
1                    5284.0  
2                     168.0  
3                     945.0  
4                     460.0  


## 5. Create merged transaction-level dataset (all features together)

This is the final ready-to-use table where each row = one purchase with all customer + article features.

In [6]:
# Merge transactions with customer features and article features
merged = transactions.merge(customers, on='customer_id', how='left')
merged = merged.merge(articles, on='article_id', how='left')

print(f'Final merged dataset shape: {merged.shape}')
print(f'\nAll columns ({len(merged.columns)}):')
for i, col in enumerate(merged.columns):
    print(f'  {i+1}. {col}')

Final merged dataset shape: (31788324, 38)

All columns (38):
  1. t_dat
  2. customer_id
  3. article_id
  4. price
  5. sales_channel_id
  6. day_of_week
  7. month
  8. is_weekend
  9. price_bracket
  10. FN
  11. Active
  12. club_member_status
  13. fashion_news_frequency
  14. age
  15. customer_total_purchases
  16. customer_unique_articles
  17. price_avg_per_customer
  18. days_since_last_purchase
  19. repeat_purchase_rate
  20. channel_preference
  21. customer_top_category
  22. age_group
  23. product_type_no
  24. product_type_name
  25. product_group_name
  26. graphical_appearance_name
  27. colour_group_name
  28. perceived_colour_value_name
  29. perceived_colour_master_name
  30. department_name
  31. index_code
  32. index_name
  33. index_group_no
  34. index_group_name
  35. section_name
  36. garment_group_name
  37. article_total_purchases
  38. article_unique_customers


## 6. Save as H&M ready dataset (parquet)

In [7]:
# Save 4 parquet files
articles.to_parquet(os.path.join(OUTPUT_DIR, 'articles.parquet'), index=False)
print(f'Saved articles.parquet: {articles.shape}')

customers.to_parquet(os.path.join(OUTPUT_DIR, 'customers.parquet'), index=False)
print(f'Saved customers.parquet: {customers.shape}')

transactions.to_parquet(os.path.join(OUTPUT_DIR, 'transactions.parquet'), index=False)
print(f'Saved transactions.parquet: {transactions.shape}')

merged.to_parquet(os.path.join(OUTPUT_DIR, 'merged.parquet'), index=False)
print(f'Saved merged.parquet: {merged.shape}')

print(f'\nAll files saved to: {OUTPUT_DIR}')

Saved articles.parquet: (105542, 17)
Saved customers.parquet: (1371980, 14)
Saved transactions.parquet: (31788324, 9)
Saved merged.parquet: (31788324, 38)

All files saved to: A:\IDEs\PyCharm\hAnd-m-PersonalizedFashionRecommendations\H&M ready dataset


In [8]:
# Verify — check files exist and show sizes
print('=== H&M ready dataset files ===')
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {f}: {size_mb:.1f} MB')

# Quick NaN check
print(f'\n=== NaN check ===')
print(f'articles NaNs: {articles.isnull().sum().sum()}')
print(f'customers NaNs: {customers.isnull().sum().sum()}')
print(f'transactions NaNs: {transactions.isnull().sum().sum()}')
print(f'merged NaNs: {merged.isnull().sum().sum()}')

=== H&M ready dataset files ===
  articles.parquet: 1.8 MB
  customers.parquet: 100.8 MB
  merged.parquet: 1412.6 MB
  transactions.parquet: 780.2 MB

=== NaN check ===
articles NaNs: 0
customers NaNs: 0
transactions NaNs: 0
merged NaNs: 0


## Summary

**Dropped 11 columns from articles** + 1 from customers (postal_code)

**Created 15 new features:**

| Feature | Where | What it captures |
|---------|-------|------------------|
| day_of_week | transactions | Weekend vs weekday behavior |
| month | transactions | Seasonality |
| is_weekend | transactions | Binary weekend flag |
| price_bracket | transactions | Cheap/mid/expensive |
| age_group | customers | Life stage bins |
| customer_total_purchases | customers | Engagement level |
| customer_unique_articles | customers | Variety preference |
| price_avg_per_customer | customers | Budget level |
| days_since_last_purchase | customers | Recency |
| repeat_purchase_rate | customers | Loyalty to same items |
| channel_preference | customers | Online vs store habit |
| customer_top_category | customers | Favorite product group |
| article_total_purchases | articles | Popularity |
| article_unique_customers | articles | Reach |
| purchase_rank | transactions | Recency position (not created — needs per-customer sorting) |

**4 parquet files saved to `H&M ready dataset/`:**
- articles.parquet (clean + 2 new article features)
- customers.parquet (clean + 8 new customer features)
- transactions.parquet (clean + 4 new time/price features)
- merged.parquet (everything joined together — ready for training)